# Module 1 · Lab 1.1 — Multi-Provider LLM Setup & Structured Outputs
## Engineering reliable LLM calls across OpenAI · Anthropic · Google · Groq

---

### Lab Overview

In a chat window you *judge* models by reading their answers. Now you **engineer** against them. Real applications don't paste replies into a chat window — they call models **programmatically**, need **structured data** they can route and store, and must stay up even when a provider has a bad day.

This lab works across **four providers** — OpenAI, Anthropic, Google and **Groq** (serving the open-source **Llama** family on custom inference hardware) — and turns ad-hoc calls into production patterns:

1. The **native shape** of all four SDKs (Responses, Messages, GenAI, OpenAI-compatible)
2. **Unified wrapper functions** so the rest of your code is provider-agnostic
3. **Temperature vs. reasoning effort** — the two knobs people confuse most
4. **Structured outputs with Pydantic** — get a *validated object*, not free text *(the headline skill)*
5. **Cost & latency** benchmarking — the data behind a vendor decision
6. A **multi-provider fallback** for production reliability

> **Judging vs. building.** Reading answers = explore & judge models *qualitatively* (free text, subjective). **This lab = build against them**: machine-readable outputs, measured latency/cost, and reliability patterns. The metric here is **schema-conformance, tokens, cost, and uptime** — not "which answer reads nicer."

### What you'll learn
- Call 4 providers natively, then behind one interface
- Control randomness (temperature) vs. depth (reasoning effort)
- Force LLM output into a **Pydantic schema** and validate it
- Estimate cost at scale and add a **fallback chain**

### Prerequisites
- API keys for **OpenAI, Anthropic, Google (Gemini),** and **Groq** (Groq has a free tier)
- Colab (recommended) or local `.env`

## 1. Install the SDKs

In [20]:
# 📦 Packages this notebook needs. On the training VM they are already installed,
# so the install line below stays commented out.
# Running somewhere else (Colab, your own laptop)? Uncomment it and run this cell once.
# !pip install -qU anthropic google-genai groq openai pandas pydantic python-dotenv

## 2. Load API keys — securely

The standard Colab-Secrets-first / `.env`-fallback loader, here with **four** keys. On Colab add them in the 🔑 **Secrets** panel; locally put them in `.env`.

In [21]:
import os

def load_keys(required):
    """Load API keys from Colab Secrets (on Colab) or a local .env file (elsewhere)."""
    try:
        from google.colab import userdata          # running on Google Colab
        for k in required:
            try:
                os.environ[k] = userdata.get(k)
            except Exception:
                pass                                 # not set in Colab Secrets
        source = "Colab Secrets"
    except ImportError:                              # running locally / non-Colab
        try:
            from dotenv import load_dotenv, find_dotenv
            load_dotenv(find_dotenv(usecwd=True))     # searches cwd upward for a .env
        except ImportError:
            pass
        source = ".env / environment"
    print(f"\U0001F511 Key source: {source}")
    for k in required:
        print(f"   {k:<22} {'\u2705' if os.environ.get(k) else '\u274C MISSING'}")

load_keys(["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY", "GROQ_API_KEY"])

🔑 Key source: .env / environment
   OPENAI_API_KEY         ✅
   ANTHROPIC_API_KEY      ✅
   GEMINI_API_KEY         ✅
   GROQ_API_KEY           ✅


## 3. Initialize all four clients

Each provider exposes a different API surface. Learn the **native** shape of each before we hide it behind a wrapper.

| Provider | Client | Primary call | Notable trait |
|---|---|---|---|
| OpenAI | `OpenAI()` | `responses.create()` / `chat.completions.create()` | Reasoning models + native structured output |
| Anthropic | `Anthropic()` | `messages.create()` | `system` is a top-level arg, not a message |
| Google | `genai.Client()` | `models.generate_content()` | Config goes in a `GenerateContentConfig` object |
| Groq | `Groq()` | `chat.completions.create()` | **OpenAI-compatible** API; very low latency |

In [22]:
import time
from openai import OpenAI
from anthropic import Anthropic
from google import genai
from google.genai import types
from groq import Groq

openai_client    = OpenAI()
anthropic_client = Anthropic()
gemini_client    = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
groq_client      = Groq()

# Pinned model ids — verify against current provider docs before each run.
M = {
    "openai_reason": "gpt-5-mini",            # reasoning model (Responses API)
    "openai_chat":   "gpt-4.1-mini",          # fast non-reasoning chat model
    "anthropic":     "claude-haiku-4-5",      # fast Claude (Sonnet 4.6 / Opus 4.8 for harder work)
    "gemini":        "gemini-2.5-flash",      # Google cost leader
    "groq":          "llama-3.3-70b-versatile",  # open-source Llama on Groq
}
print("All 4 clients initialized:", list(M.values()))

All 4 clients initialized: ['gpt-5-mini', 'gpt-4.1-mini', 'claude-haiku-4-5', 'gemini-2.5-flash', 'llama-3.3-70b-versatile']


## 4. "Hello" from each provider — one business task each

Four providers, four industries, four native SDK shapes. Read how each call is structured.

### 4a · OpenAI — Insurance claim intake 🏥 *(Responses API)*

The **Responses API** is OpenAI's newer interface and the home of reasoning models. `instructions` is the system role; `reasoning={"effort": ...}` controls how hard the model thinks.

In [23]:
resp = openai_client.responses.create(
    model=M["openai_reason"],
    instructions="You are a senior claims assistant at a health-insurance company. Be precise and empathetic.",
    input="A customer writes: 'I was hospitalized for 2 days with dengue. How do I file a cashless claim?' "
          "Give a short, numbered step-by-step.",
    reasoning={"effort": "low"},
)
print(resp.output_text)

Sorry you had to go through that — here’s a short step-by-step to file a cashless claim for a 2-day dengue hospitalization:

1. Check if the hospital is empanelled (network) with your insurer. Cashless only applies at network hospitals.  
2. Intimate your insurer immediately — preferably within 24 hours of admission or as your policy requires. You can call the toll-free number or use the insurer’s app/portal.  
3. Provide basic details: policy number, patient name, hospital name, admission date and provisional diagnosis (dengue).  
4. At the hospital, ask the admissions/TPA desk for a pre-authorization (PA) request form. The hospital usually fills this and sends it to the insurer.  
5. Submit required documents with the PA: ID, policy card, doctor’s referral/admission note, initial investigation reports (CBC, NS1/IgM if done), and any prior authorization forms. (Hospital staff often help collect these.)  
6. Await insurer approval. Once pre-authorization is approved, the insurer will s

### 4b · Anthropic — Patient-facing health FAQ 🩺 *(Messages API)*

Claude's `messages.create()` puts the **system prompt as a top-level argument** (not inside `messages`) and **requires `max_tokens`**.

In [24]:
msg = anthropic_client.messages.create(
    model=M["anthropic"],
    max_tokens=400,
    system="You are a patient-friendly hospital assistant. Use simple, reassuring language. You never give a diagnosis.",
    messages=[{"role": "user", "content": "I have a fasting blood-sugar test at 8 AM tomorrow. Can I drink water before it?"}],
)
print(msg.content[0].text)

Yes, you can drink water before your fasting blood sugar test! Water is perfectly fine and won't affect your results.

Here's what you should know:

**You CAN have:**
- Plain water (as much as you need)
- Black coffee or plain tea (without milk, cream, or sugar)

**You should AVOID:**
- Food
- Any drinks with sugar, milk, or cream
- Chewing gum (even sugar-free)
- Most medications (unless your doctor said otherwise)

**Tips for tomorrow:**
- Drink water normally up until a few hours before bed
- Try to fast for about 8-12 hours before your 8 AM test (so stop eating/drinking after midnight)
- Get a good night's sleep

If you're taking any regular medications or have questions about what you can take, it's best to check with your doctor or the lab before bed tonight.

You'll do great! These tests are routine, and the staff is very used to doing them. Let me know if you have any other questions.


### 4c · Google Gemini — E-commerce product copy 🛍️ *(GenAI SDK)*

Gemini takes the prompt as `contents` and **all settings inside a `GenerateContentConfig`** object.

In [25]:
r = gemini_client.models.generate_content(
    model=M["gemini"],
    contents="Write a 2-sentence product description for a 1L stainless-steel insulated water bottle that keeps "
             "drinks cold for 24 hours. Upbeat and benefits-first.",
    config=types.GenerateContentConfig(system_instruction="You are a creative e-commerce copywriter."),
)
print(r.text)

Experience ultimate refreshment with our 1L stainless steel insulated water bottle, engineered to keep your drinks perfectly ice-cold for a full 24 hours! Say goodbye to lukewarm sips and enjoy delicious, frosty hydration through every adventure, every single day.


### 4d · Groq — Logistics status summary 🚚 *(open-source Llama, OpenAI-compatible)*

Groq runs open-source models (Llama) on custom hardware and exposes an **OpenAI-compatible** API — if you know `chat.completions.create()`, you already know Groq.

In [26]:
r = groq_client.chat.completions.create(
    model=M["groq"],
    messages=[
        {"role": "system", "content": "You are a logistics assistant. Summarize shipment events into ONE clear customer-facing line."},
        {"role": "user",   "content": "Events: [Picked up Mumbai 10:02] [In transit Pune 14:20] "
                                       "[Out for delivery Bangalore 08:15] [Delivery attempted — customer unavailable 11:40]. "
                                       "Summarize the current status."},
    ],
)
print(r.choices[0].message.content)

Your shipment was picked up in Mumbai and is currently in Bangalore, where a delivery attempt was made at 11:40, but unfortunately, you were unavailable.


## 5. Unified wrapper functions — one interface, four providers

The native shapes differ, but our app shouldn't care. Each wrapper takes `(system, user)` and returns the **same dict**: `{provider, model, text, latency, tokens}`. This provider-agnostic pattern is what makes the comparison, cost analysis, and fallback below trivial.

In [27]:
def call_openai(system, user, model=None, temperature=0.7, max_tokens=500):
    model = model or M["openai_chat"]
    t0 = time.time()
    r = openai_client.chat.completions.create(
        model=model, temperature=temperature, max_tokens=max_tokens,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    return {"provider": "OpenAI", "model": model, "text": r.choices[0].message.content,
            "latency": round(time.time() - t0, 2), "tokens": r.usage.total_tokens}

def call_anthropic(system, user, model=None, temperature=0.7, max_tokens=500):
    model = model or M["anthropic"]
    t0 = time.time()
    r = anthropic_client.messages.create(
        model=model, max_tokens=max_tokens, temperature=temperature,
        system=system, messages=[{"role": "user", "content": user}])
    return {"provider": "Anthropic", "model": model, "text": r.content[0].text,
            "latency": round(time.time() - t0, 2), "tokens": r.usage.input_tokens + r.usage.output_tokens}

def call_gemini(system, user, model=None, temperature=0.7, max_tokens=500):
    model = model or M["gemini"]
    t0 = time.time()
    r = gemini_client.models.generate_content(
        model=model, contents=user,
        config=types.GenerateContentConfig(system_instruction=system, temperature=temperature, max_output_tokens=max_tokens))
    return {"provider": "Google", "model": model, "text": r.text,
            "latency": round(time.time() - t0, 2), "tokens": r.usage_metadata.total_token_count}

def call_groq(system, user, model=None, temperature=0.7, max_tokens=500):
    model = model or M["groq"]
    t0 = time.time()
    r = groq_client.chat.completions.create(
        model=model, temperature=temperature, max_tokens=max_tokens,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    return {"provider": "Groq", "model": model, "text": r.choices[0].message.content,
            "latency": round(time.time() - t0, 2), "tokens": r.usage.total_tokens}

PROVIDERS = {"OpenAI": call_openai, "Anthropic": call_anthropic, "Google": call_gemini, "Groq": call_groq}
print("Unified wrappers ready:", list(PROVIDERS))

Unified wrappers ready: ['OpenAI', 'Anthropic', 'Google', 'Groq']


## 6. Side-by-side comparison — same prompt, all four 🏦 *(Banking)*

**Business scenario.** A bank's app must explain a *declined transaction* to a worried customer — calmly and clearly. We send the **same** system+user prompt to all four providers and capture latency and token usage. This is the quantitative comparison that reading answers alone leaves out.

In [28]:
SYSTEM = "You are a bank's customer-support assistant. Explain clearly and calmly, in under 80 words."
USER   = ("My debit card was just declined at a petrol pump even though I have ₹40,000 in my account. "
          "Why would this happen and what should I do right now?")

results = {}
for name, fn in PROVIDERS.items():
    try:
        results[name] = fn(SYSTEM, USER)
    except Exception as e:
        results[name] = {"provider": name, "model": "-", "text": f"[ERROR] {e}", "latency": None, "tokens": None}
    print(f"\n=== {name} ({results[name]['model']}) ===")
    print(results[name]["text"])


=== OpenAI (gpt-4.1-mini) ===
Your debit card may be declined due to daily spending limits, network issues, or security blocks. Also, some petrol pumps require PIN entry or block certain transactions temporarily. Immediately, check your account for any alerts, try using another card or payment method, and contact your bank’s customer support to verify and resolve any restrictions on your card. Avoid multiple attempts to prevent it from being blocked further.

=== Anthropic (claude-haiku-4-5) ===
Several reasons could cause this:

1. **Daily limit exceeded** – Petrol pumps have transaction limits
2. **Card lock** – Security feature triggered by unusual activity
3. **Temporary glitch** – System issue at pump or bank
4. **Card expired/damaged** – Check expiry date and card condition

**Immediate steps:**
- Try another ATM or store
- Call our customer support hotline immediately
- Visit your nearest branch
- Check if card is physically damaged

We'll resolve this quickly. Have your accoun

In [29]:
import pandas as pd

df = pd.DataFrame([{"Provider": r["provider"], "Model": r["model"],
                    "Latency (s)": r["latency"], "Tokens": r["tokens"]} for r in results.values()])
print(df.to_string(index=False))
print("\nGroq usually wins latency (custom inference HW). Token counts differ — each provider tokenizes differently.")

 Provider                   Model  Latency (s)  Tokens
   OpenAI            gpt-4.1-mini         5.20     143
Anthropic        claude-haiku-4-5         2.34     197
   Google        gemini-2.5-flash         4.04     558
     Groq llama-3.3-70b-versatile         0.36     122

Groq usually wins latency (custom inference HW). Token counts differ — each provider tokenizes differently.


## 7. Temperature vs. Reasoning Effort — the two knobs people confuse

| Knob | Applies to | Controls | Range |
|---|---|---|---|
| **Temperature** | non-reasoning models (`gpt-4.1-mini`, `claude-haiku-4-5`, `llama-3.3-70b`) | randomness / creativity | `0.0` (deterministic) → `2.0` (wild) |
| **Reasoning effort** | reasoning models (`gpt-5-mini`) | depth of internal "thinking" | `low` / `medium` / `high` |

**Key rule:** reasoning models typically **don't take a temperature** — you steer them with *effort*, not randomness.

### 7a · Temperature changes *randomness* (non-reasoning model)

In [30]:
prompt = "Write a 1-sentence tagline for a budgeting app aimed at college students."
for t in (0.1, 0.7, 1.2):
    out = call_openai("You are a witty copywriter.", prompt, temperature=t)
    print(f"temp={t}: {out['text']}")

temp=0.1: Budget smart, party harder — your cash’s new BFF for college life.
temp=0.7: “Budget smarter, party harder — your campus cash companion.”
temp=1.2: Cash class: master your money, rock your college life.


### 7b · Same prompt, 3 runs at high temperature → different every time

In [31]:
for i in range(3):
    out = call_openai("You are a witty copywriter.", prompt, temperature=1.2)
    print(f"run {i+1}: {out['text']}")

run 1: Budget like a boss, so coffee, rent, and fun never have to break up.
run 2: "Budget smart, stress less—because ramen can't pay your rent."
run 3: Cash in class: ace your budget, not just your exams.


### 7c · Reasoning effort changes *depth* (reasoning model, deterministic)

In [32]:
def openai_reasoning(prompt, effort="medium"):
    t0 = time.time()
    r = openai_client.responses.create(model=M["openai_reason"], input=prompt, reasoning={"effort": effort})
    return r.output_text, round(time.time() - t0, 2)

q = ("A retail customer asks why diversification reduces risk in a mutual-fund portfolio. "
     "Explain clearly, including the basic math intuition.")
for effort in ("low", "medium", "high"):
    text, lat = openai_reasoning(q, effort)
    print(f"\n--- effort={effort}  ({lat}s) ---\n{text[:400]}...")


--- effort=low  (12.3s) ---
Short answer
Diversification reduces risk because when you hold many assets their individual ups and downs tend to offset each other. Mathematically, portfolio variance depends not only on each asset’s own variance (volatility) but also on how asset returns move together (covariances or correlations). If returns are not perfectly positively correlated, the cross‑terms lower the portfolio variance,...

--- effort=medium  (21.58s) ---
Short answer
- Diversification reduces the portion of portfolio risk that comes from individual securities (idiosyncratic risk) by “averaging out” independent ups and downs.  
- The math shows that when returns are not perfectly correlated, portfolio volatility falls as you add different securities; however you cannot eliminate market (systematic) risk that drives all assets together.

Why that ha...

--- effort=high  (52.44s) ---
Short answer
- Diversification reduces a portfolio’s risk because not all investments move the same

> **When to use which?** Deterministic compliance text → low temperature. Brainstorming marketing → high temperature. Hard multi-step analysis → a reasoning model at higher effort (slower, pricier, but deeper). Picking wrong wastes money *and* quality.

## 8. Structured outputs with Pydantic — the headline skill ⭐

So far every answer was **free text**. Production systems need **structured, machine-readable** data they can route, store, and act on. Asking a model to "return JSON" is fragile (it adds prose, breaks quotes, drifts schema). Instead we define a **Pydantic schema** and let the provider **guarantee** the output matches it.

**Business scenario — cross-industry support triage.** An inbound message could be from banking, healthcare, e-commerce, insurance... The intake system must classify each into a fixed schema so it can auto-route to the right queue and dashboard.

In [33]:
from pydantic import BaseModel, Field
from typing import Literal, List

class SupportTicket(BaseModel):
    """Structured triage for any inbound customer message."""
    category: Literal["billing", "technical", "complaint", "inquiry", "fraud_or_safety", "cancellation"] = Field(
        description="Primary intent of the message")
    urgency: Literal["low", "medium", "high", "critical"] = Field(description="How fast a human must act")
    sentiment: Literal["positive", "neutral", "negative", "angry"]
    industry: str = Field(description="Best-guess industry, e.g. 'banking', 'healthcare', 'e-commerce'")
    entities: List[str] = Field(description="Key entities: order ids, amounts, names, dates")
    summary: str = Field(description="One-sentence summary")
    recommended_action: str = Field(description="Suggested next step for the support agent")

### 8a · OpenAI `responses.parse()` — guaranteed schema → a Pydantic object

In [34]:
msg = ("My credit card was charged ₹18,250 TWICE for the same order #A4821 this morning. "
       "I need one charge refunded today — this is unacceptable.")

resp = openai_client.responses.parse(
    model=M["openai_reason"],
    input=[{"role": "system", "content": "Classify the customer message into the SupportTicket schema."},
           {"role": "user", "content": msg}],
    text_format=SupportTicket,
)
ticket = resp.output_parsed          # <-- a real, validated SupportTicket instance
print(type(ticket).__name__, "->")
print(ticket.model_dump_json(indent=2))

SupportTicket ->
{
  "category": "billing",
  "urgency": "high",
  "sentiment": "angry",
  "industry": "e-commerce",
  "entities": [
    "₹18,250",
    "Order #A4821",
    "charged twice",
    "this morning"
  ],
  "summary": "Customer reports being charged ₹18,250 twice for order #A4821 and demands one refund today.",
  "recommended_action": "Urgently verify payment records for order #A4821 and confirm the duplicate charge of ₹18,250. If confirmed, initiate an immediate refund for the duplicate charge and notify the customer with refund confirmation and expected timeline. If additional information is needed (transaction ID, last 4 digits of card), request it and escalate to the billing disputes team if refund cannot be completed today."
}


### 8b · Classify a batch of diverse, multi-industry messages

In [35]:
messages = [
    "The MRI report upload on your patient portal keeps failing with error 500. I need it before my appointment tomorrow.",   # healthcare
    "I got an OTP I never requested and someone tried to log into my account. Please block it NOW.",                          # fraud / banking
    "Just wanted to say your delivery partner was super polite and my package arrived a day early. Thank you!",               # e-commerce / positive
    "I want to cancel my annual insurance policy renewal scheduled next week and get a pro-rata refund.",                     # insurance
    "What are your customer-care hours on public holidays?",                                                                  # generic inquiry
]

rows = []
for m in messages:
    r = openai_client.responses.parse(
        model=M["openai_reason"],
        input=[{"role": "system", "content": "Classify the customer message into the SupportTicket schema."},
               {"role": "user", "content": m}],
        text_format=SupportTicket,
    )
    t = r.output_parsed
    rows.append({"industry": t.industry, "category": t.category, "urgency": t.urgency,
                 "sentiment": t.sentiment, "action": t.recommended_action[:45]})

print(pd.DataFrame(rows).to_string(index=False))

        industry        category  urgency sentiment                                        action
      healthcare       technical     high  negative Escalate to IT/engineering to investigate the
 online_services fraud_or_safety critical     angry Immediately suspend/block the user's account,
      e-commerce         inquiry      low  positive Acknowledge the customer's message with thank
       insurance    cancellation     high   neutral Ask customer to provide policy number and ver
customer service         inquiry      low   neutral Provide the customer-care hours for public ho


> Every row is a **validated object** — `ticket.urgency` is always one of the allowed values. You can now route on `urgency == "critical"` with zero string-parsing. *This* is why structured output is the backbone of agents (Module 2 onwards).

## 9. Structured output across providers — who supports what

Native structured-output support varies. Know the fallbacks:

| Provider | Native schema enforcement | Method |
|---|:--:|---|
| OpenAI | ✅ | `responses.parse(text_format=Model)` |
| Google Gemini | ✅ | `GenerateContentConfig(response_schema=Model)` → `resp.parsed` |
| Anthropic Claude | ⚠️ (via tool/prompt) | prompt for JSON, then **validate with Pydantic** |
| Groq (Llama) | ⚠️ | `response_format={"type": "json_object"}` + Pydantic validate |

The universal safety net is the same everywhere: **parse the text, then `Model.model_validate_json(...)`** so bad output fails loudly instead of silently corrupting downstream systems.

In [36]:
# Gemini — native structured output
g = gemini_client.models.generate_content(
    model=M["gemini"], contents=msg,
    config=types.GenerateContentConfig(
        system_instruction="Classify the customer message into the schema.",
        response_mime_type="application/json", response_schema=SupportTicket),
)
print("Gemini  ->", SupportTicket.model_validate_json(g.text).category)

# Claude — no native schema; prompt for JSON then validate with Pydantic
schema_hint = SupportTicket.model_json_schema()
c = anthropic_client.messages.create(
    model=M["anthropic"], max_tokens=500,
    system=f"Return ONLY valid JSON matching this JSON Schema (no prose):\n{schema_hint}",
    messages=[{"role": "user", "content": msg}],
)
import json, re
raw = c.content[0].text
raw = re.sub(r"^```(json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()  # strip code fences if present
print("Claude  ->", SupportTicket.model_validate(json.loads(raw)).category)

Gemini  -> billing
Claude  -> billing


## 10. Cost & latency at scale

The data behind a vendor decision. Plug in current per-million-token prices and estimate the **monthly** bill for a realistic workload.

In [37]:
# Approx USD per 1M tokens — REPLACE with current pricing before quoting a customer.
pricing = {
    "gpt-4.1-mini":           {"in": 0.40, "out": 1.60},
    "gpt-5-mini":             {"in": 1.10, "out": 4.40},
    "claude-haiku-4-5":       {"in": 0.80, "out": 4.00},
    "gemini-2.5-flash":       {"in": 0.15, "out": 0.60},
    "llama-3.3-70b-versatile":{"in": 0.59, "out": 0.79},
}

# Workload: 50,000 support tickets/day, ~250 input + ~150 output tokens each.
TICKETS_PER_DAY, IN_TOK, OUT_TOK = 50_000, 250, 150
days = 30
rows = []
for model, p in pricing.items():
    daily_in  = TICKETS_PER_DAY * IN_TOK  / 1_000_000
    daily_out = TICKETS_PER_DAY * OUT_TOK / 1_000_000
    monthly = (daily_in * p["in"] + daily_out * p["out"]) * days
    rows.append({"Model": model, "$/1M in": p["in"], "$/1M out": p["out"], "Est. $/month": round(monthly, 2)})

print(f"Workload: {TICKETS_PER_DAY:,} tickets/day × {days} days  ({IN_TOK} in + {OUT_TOK} out tokens each)\n")
print(pd.DataFrame(rows).sort_values("Est. $/month").to_string(index=False))

Workload: 50,000 tickets/day × 30 days  (250 in + 150 out tokens each)

                  Model  $/1M in  $/1M out  Est. $/month
       gemini-2.5-flash     0.15      0.60        191.25
llama-3.3-70b-versatile     0.59      0.79        399.00
           gpt-4.1-mini     0.40      1.60        510.00
       claude-haiku-4-5     0.80      4.00       1200.00
             gpt-5-mini     1.10      4.40       1402.50


> The cheapest model per token isn't automatically the right choice — weigh it against the **quality** you measured in §6 and each model's **caution** behaviour. Cost is one axis of three.

## 11. Production pattern: multi-provider fallback 🛡️ *(Black-Friday reliability)*

**Business scenario.** It's the peak-sale spike. Your primary provider starts returning rate-limit / 5xx errors. You **cannot** drop customer messages on the floor. A fallback chain tries providers in order and returns the first success — so a single provider's bad day doesn't take you down.

In [38]:
def classify_with_fallback(message, order=("OpenAI", "Groq", "Anthropic")):
    """Try providers in order; return the first successful response (+ which provider served it)."""
    last_error = None
    for name in order:
        try:
            out = PROVIDERS[name](
                "Reply in ONE sentence: state the category and urgency of this support message.", message)
            return {"served_by": name, "model": out["model"], "latency": out["latency"], "text": out["text"]}
        except Exception as e:
            last_error = f"{name} failed: {e}"
            print(f"  ⚠️ {last_error} → falling back...")
    raise RuntimeError(f"All providers failed. Last error: {last_error}")

result = classify_with_fallback("URGENT: my account shows a ₹0 balance but I deposited ₹50,000 this morning!")
print(f"\n✅ Served by {result['served_by']} ({result['model']}, {result['latency']}s):\n{result['text']}")


✅ Served by OpenAI (gpt-4.1-mini, 1.26s):
Category: Account Balance Issue; Urgency: Urgent.


## 12. Conclusion & what's next

| You built | Why it matters |
|---|---|
| 4 native calls + unified wrappers | One interface over four very different SDKs |
| Temperature vs. reasoning effort | The two control knobs, finally untangled |
| **Pydantic structured outputs** | Validated objects, not fragile JSON-in-text — the backbone of agents |
| Cross-provider structured output | Know each provider's native support and the universal validate-it fallback |
| Cost/latency model | Turn "which provider?" into a number |
| Fallback chain | One provider's outage ≠ your outage |

**Next — Lab 1.2: Prompting Techniques & Structured Outputs.** From raw SDK calls to LangChain's `ChatPromptTemplate` — zero- and few-shot prompting, Chain-of-Thought, Tree-of-Thought, and structured outputs inside a chain.